### IMPORT THE LIBRARIES AND TOOLS REQUIRED

In [38]:
import os
from dotenv import load_dotenv

#data ingestion libraries
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

#embeddings

from langchain_community.embeddings import JinaEmbeddings

In [39]:
load_dotenv()  # Load environment variables from .env file

True

## fetch the api keys from .env file

In [40]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

## loading the data


In [41]:
data_path = os.path.join("data","college-faq.txt")

# data ingestion

In [42]:
loader = TextLoader(data_path, encoding="utf-8", autodetect_encoding=True)

docs = loader.load()

print(f"Loaded {len(docs)} documents from {data_path}")

# Print the first 500 characters of the first document

# print(docs[0].page_content[:500])  

Loaded 1 documents from data\college-faq.txt


# splitting the data

In [43]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(docs)

print(f"Split into {len(chunks)} chunks.")

Split into 2 chunks.


# chunks and its data

In [44]:
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk.page_content[:100]}...")  # Print the first 100 characters of each chunk

Chunk 1: ==============================
COLLEGE ADMISSIONS

Applications open ...
Chunk 2: Late fee: 2 INR per book per day.


COLLEGE EXAMINATIONS
============...


## embeddings

In [45]:
vectors = JinaEmbeddings(jina_key=jina_key, model_name="jina-embedding-v2-base-en")

print("Generating embeddings for chunks...", vectors.model_name)

Generating embeddings for chunks... jina-embedding-v2-base-en


# store data in vector db

In [46]:
from langchain_community.vectorstores import FAISS  # facebook AI Similarity Search

In [47]:
 # from documnents asks for a list of documents and embeddings asks for a list of embeddings. 
 # The FAISS vector store will create an index of the embeddings and allow for efficient similarity search.

vectors = JinaEmbeddings(
   jina_api_key=jina_key,
   model_name="jina-embeddings-v2-base-en"
)

vector_store = FAISS.from_documents(chunks, vectors)

print("Vector store created with FAISS.",vector_store.index.ntotal)

Vector store created with FAISS. 2


## It will do a similarity search for every chunk

In [49]:
query = "What is the admission process for international students?"

top_match = vector_store.similarity_search(query, k=2)

print(f"Top {len(top_match)} matches for query: '{query}'")
for i, match in enumerate(top_match):
    print(f"Match {i+1}: {match.page_content}...")  # Print the first 200 characters of each match

    



Top 2 matches for query: 'What is the admission process for international students?'
Match 1: ==============================
COLLEGE ADMISSIONS

Applications open June 1 and close July 31.

Required documents:
- Application form
- Academic transcripts
- Government ID
- Passport-size photograph

Application fee: 500 INR.


COLLEGE LIBRARY

Library hours:
Monday to Saturday
8:00 AM - 8:00 PM

Students can borrow up to three books.

Borrowing period: 14 days.

Late fee: 2 INR per book per day....
Match 2: Late fee: 2 INR per book per day.


COLLEGE EXAMINATIONS

Students should arrive 30 minutes before an examination.

College ID is required.

Phones and smart watches are not permitted....


## data retrieval pipeline

In [50]:
from langchain_groq import ChatGroq

In [57]:
llm = ChatGroq(
   model = "openai/gpt-oss-120b",
   temperature = 0.4 # creativity of a model's responses. Lower values make it more deterministic, while higher values make it more creative.
)

llm.model_name

'openai/gpt-oss-120b'

In [63]:
result = llm.invoke("explain about ai in 20 words")

In [64]:
result.content

'Artificial intelligence replicates human thought, allowing computers to learn, reason, perceive, and act intelligently across many industries like healthcare, finance.'

## ai agent

## it has three things
1.brain --> llm
2.memory --> for now no memory
3.tool

In [65]:
from langchain.agents import create_agent

## TOOL

In [75]:
from langchain_core.tools import create_retriever_tool

retriever = vector_store.as_retriever(search_kwargs={"k": 2})

retriever_tool = create_retriever_tool(
    retriever,
    name="college_faq_search",
    description="""
    Search the college FAQ to answer questions about:
    - college admissions
    - application dates
    - required admission documents
    - application fees
    - library hours
    - book borrowing limits
    - borrowing periods
    - library late fees
    - examination rules
    - examination arrival time
    - required college ID
    - prohibited devices during examinations

    Use this tool whenever a student asks a question about college policies,
    admissions, library rules, or examinations.
    """
)

## AI AGENT

In [76]:
college_assistant = create_agent(
   model = llm,
   tools = [retriever_tool],
    system_prompt="""
You are a College FAQ Assistant. Your role is to answer students' questions accurately using the information provided in the retrieved context.

## Rules

1. Answer questions using ONLY the information available in the retrieved context.
2. Do NOT make up, assume, or hallucinate information.
3. If the answer is clearly present in the context, provide a direct and helpful answer.
4. If the information is not present in the context, respond:
   "I'm sorry, I don't have that information in the college FAQ."
5. Do not use outside knowledge to answer college-specific questions.
6. If the user asks multiple questions, answer each one separately when the required information is available.
7. If the retrieved context contains irrelevant information, ignore it.
8. If the information in the context is incomplete, clearly state that the available FAQ does not provide enough information.
9. Keep responses concise, clear, and student-friendly.
10. Never invent dates, fees, rules, contact details, courses, facilities, or policies.

## Examples of Expected Behavior

If the context says:

"Application fee: 500 INR."

and the student asks:

"What is the application fee?"

Answer:

"The application fee is 500 INR."

If the student asks:

"What is the hostel fee?"

and the retrieved context contains no hostel information, answer:

"I'm sorry, I don't have that information in the college FAQ."

## Context

{context}

## Student Question

{question}

## Response Instructions

* Give the answer directly.
* Use bullet points when there are multiple pieces of information.
* Preserve important numbers, dates, times, and rules exactly as provided in the context.
* Do not mention the words "context", "retrieval", "RAG", "embedding", "vector database", or "system prompt" to the student.
* Do not claim that information is available if it is not present in the retrieved context.

Answer the student's question based strictly on the provided information.

""")

## HERE WE CAN ASK QUESTIONS TO THE AGENT

In [77]:
def answer_student_question(question:str)->str:
      """
      Answer a student's question using the college FAQ assistant.
   
      Args:
         question (str): The student's question.
   
      Returns:
         str: The assistant's answer based on the retrieved context.
      """
      print(f"Student Question: {question}")
      response = college_assistant.invoke({
            "messages":[{
                  "role":"user",
                  "content":question
            }]
      })
      answer = response["messages"][-1].content #give the last message content as the answer
      print(f"Assistant Answer: {answer}")
      return answer

## WE CAN SEE THE RESULTS

In [78]:
result = answer_student_question("What is the application fee for international students?")

Student Question: What is the application fee for international students?
Assistant Answer: The application fee is **500 INR**.


In [81]:
response = college_assistant.invoke({
      "messages":[{
            "role":"user",
            "content":"What is the application fee for international students?"
      }]
})

print(response)



{'messages': [HumanMessage(content='What is the application fee for international students?', additional_kwargs={}, response_metadata={}, id='44d446df-8b65-43af-ba8c-391299a5dded'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer using context. The context placeholder is empty? We have no context provided. We need to search the FAQ using tool. Use function college_faq_search with query "application fee international students".', 'tool_calls': [{'id': 'fc_e1009ea3-6864-4302-8e7b-adf50b40409e', 'function': {'arguments': '{"query":"application fee international students"}', 'name': 'college_faq_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 647, 'total_tokens': 723, 'completion_time': 0.161159999, 'completion_tokens_details': {'reasoning_tokens': 43}, 'prompt_time': 0.039031173, 'prompt_tokens_details': None, 'queue_time': 0.406689623, 'total_time': 0.200191172}, 'model_name': 'openai/gpt-oss-1

In [85]:
print(response["messages"][-1].content)

The application fee is **500 INR**.


In [86]:
print(response["messages"][0].content)

What is the application fee for international students?


In [88]:
print(response["messages"][-1].content)


The application fee is **500 INR**.


# system message: is a set of instructions that guides the model's behavior and sets the context for its responses. 

# human message: is the student's question that is sent to the model for processing.

# ai message: is the model's response to the student's question, generated based on the system message and the retrieved context.